# Табличные данные: Pandas, Feature Engineering, CatBoost, XGBoost

Шаблоны для задач с табличными данными. Копируй нужные ячейки в свой каггл-ноутбук.

## Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, mean_squared_error, r2_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')

## Загрузка и EDA

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

TARGET = 'target'  # <-- поменять на название целевой колонки
ID_COL = 'id'      # <-- поменять на название id-колонки

print(f'Train: {train.shape}, Test: {test.shape}')
train.head()

In [ ]:
train.info()
print('---')
print('Пропуски:')
print(train.isnull().sum()[train.isnull().sum() > 0])
print('---')
print('Распределение таргета:')
print(train[TARGET].value_counts())

In [ ]:
train.describe()

In [ ]:
# Корреляции числовых признаков

# можно просто руками выбрать нужные признаки
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) <= 20:
    plt.figure(figsize=(12, 10))
    sns.heatmap(train[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

## Feature Engineering

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # Числовые пропуски -- медианой
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df[col].fillna(df[col].median(), inplace=True)

    # Категориальные пропуски -- модой
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    # или просто удалить строки с пропусками, но так можно случайно удалить всю выборку
    #df = df.dropna()
    # или же заменить пропуски на специальное значение, в зависимости от признака,
    # нужно понять какого вида пропуски

    # Label encoding для категорий
    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    # --- Дописать свои фичи ниже ---
    # df['new_feature'] = df['col1'] * df['col2']
    # df['ratio'] = df['col1'] / (df['col2'] + 1)

    return df

train = feature_engineering(train)
test = feature_engineering(test)

In [ ]:
features = [c for c in train.columns if c not in [TARGET, ID_COL]]
X = train[features]
y = train[TARGET]
X_test = test[features]

print(f'Features: {len(features)}, X: {X.shape}, X_test: {X_test.shape}')

## CatBoost

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

# --- КЛАССИФИКАЦИЯ ---
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = CatBoostClassifier(
    # изначально стоит просто запустить из коробки
    # iterations=2000,
    # learning_rate=0.05,
    # depth=6,

    eval_metric='F1',           # F1 / AUC / Accuracy / Logloss
    early_stopping_rounds=100,
    verbose=200,
    random_seed=42,
    # cat_features=cat_cols,    # если есть категориальные (до label encoding)
)
model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

val_preds = model.predict(X_val)
print(classification_report(y_val, val_preds))

In [ ]:
# --- РЕГРЕССИЯ ---
# model = CatBoostRegressor(
#     iterations=2000,
#     learning_rate=0.05,
#     depth=6,
#     eval_metric='RMSE',
#     early_stopping_rounds=100,
#     verbose=200,
#     random_seed=42,
# )
# model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
# val_preds = model.predict(X_val)
# print(f'RMSE: {mean_squared_error(y_val, val_preds, squared=False):.4f}')
# print(f'R2:   {r2_score(y_val, val_preds):.4f}')

## XGBoost

In [ ]:
import xgboost as xgb

model_xgb = xgb.XGBClassifier(
    # тут из коробки не так хорошо, тут подбираем гиперпараметры или берем с потолка
    # над каждым из вас перед туром напишем идеальные гиперпараметры
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
)
model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)

val_preds_xgb = model_xgb.predict(X_val)
print(classification_report(y_val, val_preds_xgb))

## Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature': features,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, max(6, len(features) * 0.3)))
sns.barplot(data=fi.head(30), x='importance', y='feature')
plt.title('Top Feature Importances (CatBoost)')
plt.tight_layout()
plt.show()

## Submission

In [ ]:
test_preds = model.predict(X_test)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds
})
submission.to_csv('submission.csv', index=False)
submission.head()